# Object timeline: demo on the running example

Concept: `claude_notes/Object Timeline (draft).md` (outside this repo; a draft, not approved yet).
Code: module `ocelescope_module_location_based_ocpm.timeline.object_timeline`; this notebook only imports and calls its functions, so a change in the package shows up here on the next run.
Further detail: the docstrings (e.g. `help(band_segments)`) and `docs/resources/object-timeline.md`.

What the notebook does:
1. loads the inputs: the location mapping of one level (saved by `location_derivation.ipynb`), the location context ($agg$, $V$), the processing operations $P$ and the consumed/produced handling units of the transformation events,
2. prepares the relations step by step (high-level locations, vehicles, processing, roles, markers, order, positions, pickup disagreements),
3. runs the same with one call (`timeline_relations`),
4. derives what the figure draws: the band stretches, the same-timestamp groups, the shared events and the lifespans,
5. draws the timeline on both axes (event order, real time),
6. shows how events with the same timestamp are handled,
7. shows the edge cases of `data/timeline_example/`,
8. saves the figures as HTML files.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/object_timeline.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$.

In [ ]:
from datetime import datetime
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import load_level_mapping
from ocelescope_module_location_based_ocpm.timeline.object_timeline import (
    add_high_level_locations,
    add_markers,
    add_pickup_differs,
    add_positions,
    add_processing,
    add_transformation_roles,
    add_vehicles,
    band_segments,
    hu_order,
    lifespans,
    object_timeline,
    shared_events,
    tie_groups,
    timeline_relations,
)
from ocelescope_module_location_based_ocpm.visits.context import location_context

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"
INPUTS = EXAMPLE / "inputs"
EDGE = ROOT / "data" / "timeline_example"
OUTPUT = ROOT / "notebooks" / "output"

pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250);

## 1. Inputs

### Location mapping: `load_level_mapping(folder, level)`

The current and previous locations of one level, as saved by `location_derivation.ipynb` in `data/running_example/derived/locations_level_{k}.csv`: columns event, hu, object_type, time, activity, is_movement ($e \in M$), is_transformation ($e \in T$), `loc` ($= loc_k$) and `ploc` ($= ploc_k$).
Here: Level 3, the selected level of the running example.

In [ ]:
LEVEL = 3
mapping = load_level_mapping(EXAMPLE / "derived", LEVEL)
mapping

### Location context: `location_context(table)`

$agg$ (partial; unmapped locations are their own high-level location) and $V$, from `inputs/locations.csv` (see `trace_figure.ipynb` for the parameters).
Running example: A1, A2 → A; B1 → B; T unmapped; $V = \{T\}$.

In [ ]:
context = location_context(pl.read_csv(INPUTS / "locations.csv"))
context

### Processing operations $P$ and transformation roles

- `inputs/processing_events.csv`: one column `event`, the processing operations $P$ (running example: cast slab e1, roll e3, label e4, cut e15).
- `inputs/transformation_roles.csv`: one row per consumed or produced handling unit of a transformation event: columns event, hu, role (`consumed` or `produced`); the same as the e2o qualifiers of the log.
  A handling unit consumed and produced at the same event has two rows.

Both formats are drafts (written by `make_running_example.py`) until the inputs are decided.

In [ ]:
PROCESSING = pl.read_csv(INPUTS / "processing_events.csv")["event"].to_list()
ROLES = pl.read_csv(INPUTS / "transformation_roles.csv")
print("P =", PROCESSING)
ROLES

## 2. Preparing the relations, step by step

Each step takes a relations frame and returns it with columns added.

### High-level locations: `add_high_level_locations(relations, agg=None)`

Adds `high_level` $= \widehat{hl}((e,hu)) = agg(loc((e,hu)))$ and `prev_high_level` $= \widehat{phl}((e,hu)) = agg(ploc((e,hu)))$, ⊥ where the location is ⊥.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with `loc` and `ploc` | |
| `agg` | dict low-level → high-level location; unmapped locations stay as they are (identity) | identity for all |

How it works: a polars `replace` on both columns.

In [ ]:
rel = add_high_level_locations(mapping, context.agg)
rel.select("hu", "event", "loc", "ploc", "high_level", "prev_high_level")

### Vehicles: `add_vehicles(relations, vehicles=())`

Adds `is_vehicle` ($loc((e,hu)) \in V$) and `prev_is_vehicle` ($ploc((e,hu)) \in V$); false for ⊥.
Band stretches on vehicles are hatched.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with `loc` and `ploc` | |
| `vehicles` | the set $V$ | empty |

In [ ]:
rel = add_vehicles(rel, context.vehicles)
rel.filter(pl.col("hu") == "c1").select("event", "loc", "ploc", "is_vehicle", "prev_is_vehicle")

### Processing operations: `add_processing(relations, processing=None)`

Adds `is_processing` ($e \in P$).

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with `event` and `is_transformation` | |
| `processing` | event ids of $P$; an empty collection marks none | `None` = the transformation events $T$ (Nina 2026-10-31: if no $P$ is provided, only $T$) |

In [ ]:
rel = add_processing(rel, PROCESSING)
rel.filter(pl.col("is_processing")).select("hu", "event", "activity", "is_transformation", "is_processing")

### Consumed and produced: `add_transformation_roles(relations, roles=None)`

Adds `is_consumed` ($hu \in cons(e)$) and `is_produced` ($hu \in prod(e)$).
Roles only count at transformation events ($cons$ and $prod$ are defined on $T$ only); a row of `roles` for another event is ignored.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with `event`, `hu`, `is_transformation` | |
| `roles` | table event, hu, role (`consumed`/`produced`) | `None` = no roles |

How it works: the roles are grouped per (event, hu) into two booleans and left-joined; missing = false.

In [ ]:
rel = add_transformation_roles(rel, ROLES)
rel.filter(pl.col("is_transformation")).select("hu", "event", "activity", "is_consumed", "is_produced")

### Markers: `add_markers(relations)`

Adds the marker shape and the role of every relation:

| column | value | when | drawn as |
|---|---|---|---|
| `marker` | `processing` | $e \in P$ | star (above the band; on the band if also $e \in M$) |
| | `movement` | $e \in M \setminus P$ | on the band: ▲ produced, ▼ consumed, otherwise a vertical tick (a small open diamond if the pickup location differs, see `add_pickup_differs`) |
| | `other` | otherwise | small circle above the band |
| `role` | `consumed and produced`, `produced`, `consumed`, null | from `is_consumed`, `is_produced` | colour: amber, green, pink, dark grey |

Example: e3 (roll) is a pink star for s1 (consumed) and a green star for c1 and c2 (produced).

In [ ]:
rel = add_markers(rel)
rel.select("hu", "event", "activity", "is_movement", "is_processing", "marker", "role")

### Order of the rows: `hu_order(relations, handling_units=None)`

Handling unit → row (0 = top).

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with `hu` and `time` | |
| `handling_units` | handling units to show, in this order; ids without relations are dropped | `None` = all, by the time of their first event, then by id |
| `max_handling_units` | show at most this many (the first ones), with a warning naming how many were cut; `None` = no limit | 50 |

In [ ]:
print(hu_order(rel))
print(hu_order(rel, ["c3", "c1", "unknown"]))

### Positions: `add_positions(relations, axis="event order", spread=0.5)`

Needs the column `hu_order`.
Order of the relations: by time; among events with the same timestamp **movements first** (an event at the same timestamp as a movement counts as after it, as in Level 1 of the location derivation), then by event id.
Adds `position` (index in the handling unit's trace in this order, from 1) and `x`:

| `axis` | x |
|---|---|
| `"event order"` (default) | rank of the event among all shown events in this order: every event has its own x, a shared event has one x |
| `"time"` | the timestamp; tied events are placed side by side in this order around it, together spanning at most `spread` times the smallest gap between timestamps (section 6) |

In [ ]:
order = hu_order(rel)
rel = add_positions(rel.with_columns(pl.col("hu").replace_strict(order, return_dtype=pl.Int64).alias("hu_order")))
rel.select("hu", "hu_order", "event", "time", "position", "x")

### Pickup disagreements: `add_pickup_differs(relations)`

The band shows $loc$ only. Where a movement's provided pickup location disagrees with where the handling unit was, the movement is marked instead (a small open diamond on the band; details in the hover).

Adds `prev_loc` ($loc$ of the handling unit's previous relation, in the order above) and `pickup_differs`: true for $e \in M$ with $ploc((e,hu)) \ne \bot$, `prev_loc` $\ne \bot$ and $ploc((e,hu)) \ne$ `prev_loc`.
For a movement, the previous relation always has an earlier timestamp (movements come first among tied events, and by RQ3 a handling unit has at most one movement per timestamp).

Example: e13 picks c1 up at A1, but c1 was at A2 after e11.

In [ ]:
rel = add_pickup_differs(rel)
rel.filter(pl.col("is_movement")).select("hu", "event", "activity", "ploc", "prev_loc", "loc", "pickup_differs")

## 3. All in one call: `timeline_relations(relations, handling_units=None, agg=None, vehicles=(), processing=None, roles=None, axis="event order", max_handling_units=50)`

Selects and orders the handling units and runs all steps of section 2.

| parameter | meaning | default |
|---|---|---|
| `relations` | location mapping (one row per hu-event relation) | |
| `handling_units` | handling units to show, top to bottom | all, by first event |
| `agg`, `vehicles` | location context | identity, none |
| `processing` | $P$ | $T$ |
| `roles` | consumed/produced table | none |
| `axis` | `"event order"` or `"time"` | `"event order"` |
| `max_handling_units` | see `hu_order` | 50 |

In [ ]:
relations = timeline_relations(mapping, None, context.agg, context.vehicles, PROCESSING, ROLES)
relations.equals(rel.select(relations.columns))

## 4. What the figure draws

### Band stretches: `band_segments(relations, level="low", reach=None)`

The coloured stretches of one half of the band (concept draft, section 5), per handling unit with relations $r_1, \dots, r_n$:
1. after $r_j$: $loc(r_j)$ up to $r_{j+1}$ (only $loc$);
2. lead-in: the location just before the first timestamp for `reach` before $r_1$, if known and the handling unit is not produced there ($ploc$ of the movement at that timestamp if there is one, else of any relation there; `timestamp_before`);
3. end: $loc(r_n)$ for `reach` after $r_n$, unless the handling unit is consumed at $r_n$;
4. ⊥ stretches are left out; touching stretches with the same location and vehicle status are merged.

| parameter | meaning | default |
|---|---|---|
| `relations` | output of `timeline_relations` | |
| `level` | `"low"` ($loc$, $ploc$) or `"high"` ($\widehat{hl}$, $\widehat{phl}$; vehicle status from the low-level location) | `"low"` |
| `reach` | length $\delta$ of lead-in and continuation, in x units (positions, or a `timedelta` on the time axis) | `continuation(relations)`: 0.3 positions, or 1 % of the time span (at least 30 seconds) |

Returns one row per stretch: hu, hu_order, level, location, is_vehicle, x0, x1.

Example c1: A2 [9, 13], then T hatched; open end after e14 (14.3). s1 ends at e3 and c2 at e15 (consumed); c5 has no stretch (never located).

In [ ]:
band_segments(relations, "low")

In [ ]:
band_segments(relations, "high")

### Same timestamp, shared events, lifespans

- `tie_groups(relations)`: timestamps with several shown events: time, x0, x1, events, n (the grey boxes). The running example has none (section 6 has an example).
- `shared_events(relations)`: events of several shown handling units: event, x, hus, n (the dotted lines).
- `lifespans(relations)`: first and last x per handling unit (the lifespan lines).

In [ ]:
tie_groups(relations)

In [ ]:
shared_events(relations)

In [ ]:
lifespans(relations)

## 5. The figure: `object_timeline(relations, show_high_level=True, show_activity_labels=True, mark_ties=True, link_shared_events=True, reach=None, title=None, width=1200)`

| parameter | meaning | default |
|---|---|---|
| `relations` | output of `timeline_relations` | |
| `show_high_level` | lower half of the band shows $\widehat{hl}$; otherwise the band is $loc$ only | `True` |
| `show_activity_labels` | activity name at an angle above every marker not on the band | `True` |
| `mark_ties` | grey boxes behind events with the same timestamp | `True` |
| `link_shared_events` | dotted lines joining the markers of shared events | `True` |
| `reach` | see `band_segments` | default |
| `title`, `width` | figure title, width in pixels | none, 1200 |

Hover on a marker for activity, event, kinds, role, time, $ploc \to loc$ and the high-level locations; hover on a band stretch for its location.

In [ ]:
figure = object_timeline(relations, title=f"Running example, Level {LEVEL}, x = event order")
figure.show()

Real time: gaps between events show durations; the band stretches keep their meaning.

In [ ]:
relations_time = timeline_relations(mapping, None, context.agg, context.vehicles, PROCESSING, ROLES, axis="time")
figure_time = object_timeline(relations_time, title=f"Running example, Level {LEVEL}, x = time")
figure_time.show()

A selection of handling units, in a given order (the lineage of s1: s1 → c1, c2 → c4):

In [ ]:
lineage = timeline_relations(mapping, ["s1", "c2", "c4", "c1"], context.agg, context.vehicles, PROCESSING, ROLES)
object_timeline(lineage, title="Lineage of s1").show()

## 6. Events with the same timestamp

The running example has no events with the same timestamp, so here three events are moved (only in this notebook, on the loaded mapping): e4 to 03:00 (tied with e3), e14 to 13:00 (tied with e13), e16 to 15:00 (tied with e15).
At Level 3 this keeps the locations consistent (tied relations of a handling unit have the same $loc$).

How ties are handled (concept draft, section 7):
1. every event has its own x: by time, among tied events movements first, then event id (`add_positions`); on the event-order axis tied events get neighbouring positions; on the time axis they are spread around their timestamp (over at most half of the smallest gap between timestamps), and the hover shows the true time;
2. a light grey box marks each group of tied events over the whole height ("same timestamp" in the legend);
3. at Levels 1 to 3 all tied relations of a handling unit have the same $loc$, so the band does not change inside a box.

In [ ]:
TIED_TIMES = {"e4": datetime(2026, 1, 1, 3), "e14": datetime(2026, 1, 1, 13), "e16": datetime(2026, 1, 1, 15)}
tied = mapping.with_columns(
    pl.col("event").replace_strict(TIED_TIMES, default=None, return_dtype=pl.Datetime("us")).fill_null(pl.col("time")).alias("time")
)
tied_relations = timeline_relations(tied, None, context.agg, context.vehicles, PROCESSING, ROLES)
tie_groups(tied_relations)

In [ ]:
object_timeline(tied_relations, title="Ties, x = event order").show()

In [ ]:
tied_time = timeline_relations(tied, None, context.agg, context.vehicles, PROCESSING, ROLES, axis="time")
print(tie_groups(tied_time))
object_timeline(tied_time, title="Ties, x = time").show()

The bands are the same as without ties (only the x values change), since tied relations of a handling unit have the same location at Level 3:

In [ ]:
band_segments(tied_relations).filter(pl.col("hu") == "c1")

## 7. Edge cases: `data/timeline_example/`

A small hand-written location mapping (see its `README.md`, generated by `make_timeline_example.py`) with what the running example lacks: a lead-in (h1 comes from X1), a consumption at a movement (h1 at x3, the band ends) and a production at a movement (h2 at x4, no lead-in), a processing operation that consumes and produces (x5), ties of one handling unit (x5/x6, x7/x8) and of different ones, a movement first inside a tie although its id is larger (x8 before x7), a pickup disagreement (x8: Y3, but h2 was at Y2) and a gap at ⊥ (h3 at x9).
Inside the tie at 03:00 both events are movements, so the event id decides: "x10" comes before "x3".

In [ ]:
edge_mapping = pl.read_csv(EDGE / "relations.csv", try_parse_dates=True)
edge_context = location_context(pl.read_csv(EDGE / "locations.csv"))
edge = timeline_relations(
    edge_mapping,
    agg=edge_context.agg,
    vehicles=edge_context.vehicles,
    processing=pl.read_csv(EDGE / "processing_events.csv")["event"].to_list(),
    roles=pl.read_csv(EDGE / "transformation_roles.csv"),
)
edge.select("hu", "event", "time", "x", "loc", "ploc", "prev_loc", "pickup_differs", "marker", "role")

In [ ]:
band_segments(edge, "low")

In [ ]:
figure_edge = object_timeline(edge, title="Edge cases")
figure_edge.show()

## 8. Save the figures

`figure.write_html(path)` saves an interactive file; with `include_plotlyjs="cdn"` the file is small but needs internet to open.
The files go to `notebooks/output/` (not committed).

In [ ]:
OUTPUT.mkdir(exist_ok=True)
figure.write_html(OUTPUT / "object-timeline-event-order.html", include_plotlyjs="cdn")
figure_time.write_html(OUTPUT / "object-timeline-time.html", include_plotlyjs="cdn")
figure_edge.write_html(OUTPUT / "object-timeline-edge-cases.html", include_plotlyjs="cdn")